# 05 — RTO analysis and the H1 decomposition

Phase 3 deliverables B and C. The centrepiece is C: **how much of the
COD–RTO gap is causation, and how much is selection?**

Mirrors `sql/11_economics.sql` and `sql/12_hypotheses.sql`.
`scripts/05_crosscheck.py` asserts all 56 metrics agree across SQL and Python,
and asserts that the `analyst` role producing this analysis is denied on schema
`truth` — so nothing here can see the answer it is estimating.


In [ ]:
import sys, json; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src.analysis import funnel as F, h1_decomposition as H

TABLES = F.load_tables()
POP    = pd.read_parquet(ROOT / 'data/processed/h1_population.parquet')
TRUTH  = json.loads((ROOT / 'data/truth/_truth.json').read_text())
EFF    = TRUTH['planted_causal_effects']['cod_on_rto']
AME, NAIVE = EFF['average_marginal_effect_pp'], EFF['naive_observed_gap_pp']
print(f'population {len(POP):,} (shipped AND NOT censored)')
print(f'truth AME {AME:.2f}pp | naive {NAIVE:.2f}pp | selection {NAIVE-AME:.2f}pp')


## B — the waterfall


In [ ]:
w = F.waterfall(TABLES)
pd.Series(w)


In [ ]:
F.avoidability(TABLES)


Addressable is **measured** at 61.44% of cost. Phase 1 §7.2 assumed 65% —
logged as a fourth missed prior alongside H2, H3 and H11.


## C — the four estimates

Run in order of how much each controls for. Every one must satisfy
`AME < estimate < naive`; none may reach the AME, because the latents that
drive the confounding are unobservable by construction.


In [ ]:
r1 = H.raw_crosstab(POP)
r2 = H.stratified(POP)
r3 = H.logistic_adjusted(POP)
r4 = H.propensity_matched(POP)

pd.DataFrame([
    {'method': r['method'], 'estimate_pp': round(r['estimate_pp'], 3),
     'recovers': f"{(NAIVE - r['estimate_pp']) / (NAIVE - AME):.1%}",
     'controls': r['controls']}
    for r in (r1, r2, r3, r4)
] + [{'method': 'TRUTH (unobservable)', 'estimate_pp': round(AME, 3),
      'recovers': '100.0%', 'controls': 'the latents themselves'}])


### The stratified cells


In [ ]:
r2['table']


### The estimand matters by 3.7pp

ATT is the one to quote: the truth file's AME is averaged over the shipped COD
population, so it is an effect on the *treated*. The prepaid-weighted figure
sits closer to the truth and is answering the mirror-image question.


In [ ]:
pd.Series({
    'ATT (COD-weighted)':     r2['estimate_att_cod_weighted_pp'],
    'ATE (pooled)':           r2['estimate_ate_pooled_weighted_pp'],
    'ATU (prepaid-weighted)': r2['estimate_atu_prepaid_weighted_pp'],
    'truth AME':              AME,
}).round(3)


### GT-03 — the adjustment must move toward the truth without arriving


In [ ]:
pd.DataFrame([{'method': r['method'], **H.gt_03(r['estimate_pp'], AME, NAIVE)}
              for r in (r2, r3, r4)]).round(4)


The ordering condition passes for all three. The **magnitude** floor (0.35 of
the selection component must survive) fails for the regression and the match.

The cause is structural, not a leak: decision A11 generates pre-window history
**from the latents**, so `pit_cod_share` is a direct observable consequence of
the unobservable confounder. See §C.3 of `reports/phase3_findings.md`. It is
logged for a ruling rather than fixed by dropping the offending feature —
that would be selecting a specification to pass a test.


## The honest ceiling

The best estimate still overstates the true effect by 6.8%, with 41 confounders
and a favourable proxy structure. An analyst without the truth file would have
**no way to know** the residual was 0.68pp rather than 5pp.

No observational method can identify the causal share. Only the Phase 6
partial-payment experiment can.
